**Машинное обучение в экономике**

**Семинар 1. Байесовские сети**

Установка библиотек: чтобы установить библиотеку, необходимо удалить знак комментария `#`.

In [ ]:
# !python.exe -m pip install --upgrade pip
# !pip install numpy==1.26.4
# !pip install pandas # 1.5.3
# !pip install scikit-learn
# !pip install d3blocks==1.4.9
# !pip install pgmpy==0.1.20
# !pip install bnlearn==0.10.1
# exit()

Подключение библиотек

In [ ]:
import numpy  as np                                    # базовые операции с массивами
import pandas as pd                                    # базовые операции с датафреймами
import scipy  as scipy                                 # базовые операции с распределениями
import random                                          # случайные числа
import bnlearn                                         # Байесовские сети
from sklearn.naive_bayes     import CategoricalNB      # наивный Байесовский классификатор
from sklearn.model_selection import train_test_split   # разделение выборки на
                                                       # обучающую и тестовую
from sklearn.model_selection import cross_val_score    # кросс-валидация
from sklearn.model_selection import KFold              # разбиение на части (folds)
from sklearn.utils           import shuffle            # случайная перестановка

**Рассматриваемые методы**

1. Байесовский классификатор (b - Bayes)
2. Наивный Байесовский классификатор (nb - naive Bayes)
3. Байесовские сети (bn - Bayesian network)

**Рассматриваемые техники сравнения качества моделей**

1. Внутривыборочная точность прогноза
2. Вневыборочная точность прогноза
3. Кросс-валидация

**Генерация данных** 🐰

In [ ]:
# Для воспроизводимости
np.random.seed(123)

# Число наблюдений
n = 1000

# Названия переменных
features_names = ["age", "educ", "married", "nchildren", "work", "income"]

# Число признаков
m = len(features_names)

# Ковариации
sigma = np.empty(shape=(m, m), dtype = 'object')
sigma = pd.DataFrame(np.ones(shape=(m, m), dtype = 'object'),
                     index   = features_names,
                     columns = features_names)
  # age
sigma.loc["age", "educ"]      = 0.3
sigma.loc["age", "married"]   = 0.4
sigma.loc["age", "nchildren"] = 0.5
sigma.loc["age", "work"]      = 0.3
sigma.loc["age", "income"]    = 0.4
  # educ
sigma.loc["educ", "married"]   = 0.3
sigma.loc["educ", "nchildren"] = -0.1
sigma.loc["educ", "work"]      = 0.5
sigma.loc["educ", "income"]    = 0.6
  # married
sigma.loc["married", "nchildren"] = 0.5
sigma.loc["married", "work"]      = 0.3
sigma.loc["married", "income"]    = 0.2
  # nchildren
sigma.loc["nchildren", "work"]   = 0.1
sigma.loc["nchildren", "income"] = 0.1
  # work
sigma.loc["work", "income"] = 0.7

# Делаем нижнюю и верхнюю части ковариационной
# матрицы симметричными
for i in range(0, m):
  for j in range(0, i):
    sigma.iloc[i, j] = sigma.iloc[j, i]

# Генирируем данные из многомерного нормального распределения
df         = np.random.multivariate_normal(mean = np.zeros(m),
                                           cov  = sigma,
                                           size = n)
df         = pd.DataFrame(df)
df.columns = features_names

# Приводим данные к реалистичному виду
  # age
df["age"] = pd.cut(df.loc[:, "age"],
                   bins = scipy.stats.norm.ppf
                     (
                     np.linspace(start = 0, stop  = 1, num = 42),
                                 loc   = 0, scale = 1
                     ),
                   labels = range(20, 61)
                  )
  # educ
df["educ"] = pd.cut(df.loc[:, "educ"],
                    bins = scipy.stats.norm.ppf
                      (
                        [0, 0.6, 0.9, 1],
                        loc = 0, scale = 1
                      ),
                      labels = range(1, 4)
                   )
  # married
df["married"] = pd.cut(df.loc[:, "married"],
                       bins = scipy.stats.norm.ppf
                         (
                           [0, 0.4, 1],
                           loc = 0, scale = 1
                         ),
                       labels = range(0, 2)
                      )
  # nchildren
df["nchildren"] = pd.cut(df.loc[:, "nchildren"],
                         bins = scipy.stats.norm.ppf(
                           np.concatenate
                             (
                               (
                                 [0],
                                 scipy.stats.poisson.cdf(k  = range(0, 10),
                                                         mu = 2),
                                 [1]
                               )
                             ),
                           loc    = 0,
                           scale  =  1),
                           labels = range(0, 11)
                        )
  # work
df["work"] = pd.cut(df.loc[:, "work"],
                    bins = scipy.stats.norm.ppf
                             (
                              [0, 0.3, 1],
                              loc = 0, scale = 1
                             ),
                    labels = range(0, 2)
                    )
  # income
df["income"] = np.round(
                         scipy.stats.expon.ppf
                           (
                             scipy.stats.norm.cdf(df["income"]), scale = 10
                           )
                       ) + 10

# Конвертируем данные в нужный формат
df = df.astype(float)

# Симулируем дефолт
default_li = (
              2                              -
              0.03    * df["age"]            +
              0.00005 * df["age"] ** 2       -
              0.6     * (df["educ"] == 2.0)  -
              0.8     * (df["educ"] == 3.0)  +
              0.5     * df["married"]        +
              0.2     * df["nchildren"]      -
              0.6     * df["work"]           -
              0.5     * np.log(df["income"]) +
              0.02    * np.log(df["income"]) * df["nchildren"]
             )
default_prob  = scipy.stats.t.cdf(default_li, df = 5)
df["default"] = (default_prob >= 0.5).astype(float)

# Доля дефолтов
print({'defaults': np.mean(df["default"])})

# Корреляционная матрица
print(df.astype(float).corr(method = 'pearson'))

# Данные
print(df)

{'defaults': 0.339}
                age      educ   married  nchildren      work    income  \
age        1.000000  0.229076  0.285371   0.473610  0.214207  0.326508   
educ       0.229076  1.000000  0.174396  -0.134767  0.271294  0.476841   
married    0.285371  0.174396  1.000000   0.362492  0.156524  0.092141   
nchildren  0.473610 -0.134767  0.362492   1.000000  0.067413  0.069018   
work       0.214207  0.271294  0.156524   0.067413  1.000000  0.420284   
income     0.326508  0.476841  0.092141   0.069018  0.420284  1.000000   
default   -0.271244 -0.476018  0.130545   0.281258 -0.534532 -0.431395   

            default  
age       -0.271244  
educ      -0.476018  
married    0.130545  
nchildren  0.281258  
work      -0.534532  
income    -0.431395  
default    1.000000  
      age  educ  married  nchildren  work  income  default
0    51.0   1.0      1.0        4.0   1.0    30.0      0.0
1    59.0   2.0      1.0        3.0   1.0    57.0      0.0
2    20.0   1.0      0.0        1.

**Первичный анализ данных** 🐱

**Цели** ⭐

*   Посмотреть описательные статистики.  
*   Оценить маржинальные и совместные вероятности.

Описание данных

*   `default`   - дефолт: 0 - не наступил, 1 - наступил.
*   `age`       - возраст в годах.
*   `educ`      - уровень образовнания: 1 - среднее общее, 2 - среднее специальное, 3 - высшее.
*   `married`   - семейный статус: 0 - не в браке, 1 - в браке.
*   `nchildren` - количество детей.
*   `work`      - занятость: 0 - не работает, 1 - работает.
*   `income`    - доход в тысячах рублей.



In [ ]:
# Посмотрим первые несколько строк в данных
df.head(10)

Оценим маржинальные вероятности брака $\text{P}(\text{Married} = 1)$ и занятости $\text{P}(\text{Work} = 1)$ как выборочные средние (доли):

*   $\hat{\text{P}}(\text{Married} = 1) = \frac{1}{n}\sum\limits_{i=1}^{n} \text{Married}_{i}$
*   $\hat{\text{P}}(\text{Work} = 1) = \frac{1}{n}\sum\limits_{i=1}^{n} \text{Work}_{i}$


Техническое примечание ⚡

Команда `df["название столбца"]` позволяет выбрать определенный столбец в датафрейме `df`.

Функция `np.mean()` считает среднее значение. Например, `np.mean(df["название столбца"])` считает среднее значение по столбцу.

Конструктор `pd.DataFrame()` создает датафрейм и использует следующие основные аргументы:

*   `data` - данные.
*   `index` - имена строк.
*   `columns` - имена столбцов.



In [ ]:
# Для простоты построим модель для прогнозирования дефолта, рассматривая
# в качестве признаков лишь семейный и трудовой статусы
p_married = np.mean(df["married"])  # оценка P(Брак = 1)
p_work    = np.mean(df["work"])     # оценка P(Работа = 1)

# Посмотрим на результат
print(pd.DataFrame(data    = [p_married, p_work],               # данные
                   index   = ['P(Брак = 1)', 'P(Работа = 1)'],  # имена строк
                   columns = ['Оценка']))                       # имена столбцов

Индивидов можно разделить на 4 группы
1. В браке и работающие: $\text{Married }= 1, \text{Work} = 1$
2. В браке и безработные: $\text{Married }= 1, \text{Work} = 0$
3. Холостые и работающие: $\text{Married }= 0, \text{Work} = 1$
4. Холостые и безработные: $\text{Married} = 0, \text{Work} = 0$

Изучим, сколько индивидов в наших данных относится к каждой из групп

In [ ]:
# Рассмотрим совместное распределение брака и работы
counts_married_work = pd.crosstab(df["married"], df["work"])
print(counts_married_work)

In [ ]:
# Сохраним число наблюдений
n = df.index.size

Рассмотрим совместные вероятности:

$$P(\text{Married} = a, \text{Work} = b),\text{ где }a,b\in\{0,1\}$$

Например:

*  $P(\text{Married} = 1, \text{Work} = 0)$ оценивается как доля безработных индивидов, состоящих в браке:
$\hat{P}(\text{Married} = 1, \text{Work} = 0)=\frac{1}{n}\sum\limits_{i=1}^{n}I(\text{Married}_{i}=1,\text{Work}_{i}=0)$
*  $P(\text{Married} = 0, \text{Work} = 1)$ оценивается как доля холостых работающих индивидов:$\hat{P}(\text{Married} = 1, \text{Work} = 0)=\frac{1}{n}\sum\limits_{i=1}^{n}I(\text{Married}_{i}=0,\text{Work}_{i}=1)$

Где функция индикатор задается следующим образом:

$$I(условие)=\begin{cases}1\text{, если условие соблюдено}\\0\text{, в противном случае}\end{cases}$$

In [ ]:
# Посмотрим доли (оценки совместных вероятностей)
print(counts_married_work / n)

**Байесовский классификатор: ручная реализация на простом примере** 🐱

**Цели** ⭐

*   Оценить условную вероятность дефолта для каждого индивида в выборке c помощью Байесовского классификатора.  
*   Используя оцененные вероятности спрогнозировать дефолт.
*   Посчитать точность прогноза $\text{ACC}$.


Условные вероятности:

$$P(\text{Defaul} = c|\text{Married} = a, \text{Work} = b),\text{ где }a,b,c\in\{0,1\}$$

Например:

*   $P(\text{Defaul} = 1|\text{Married} = 1, \text{Work} = 0)$ оценивается как доля дефолтов среди безработных индивидов, состоящих в браке.
*   $P(\text{Defaul} = 0|\text{Married} = 0, \text{Work} = 1)$ оценивается как доля индивидов без дефолтов среди холостых работающих индивидов.

Для краткости обозначим:

$$p_{ab} = \hat{P}(\text{Defaul} = 1|\text{Married} = a, \text{Work} = b)$$

Техническое примечание ⚡

Команда
`df.loc[условие, "название столбца"]` позволяет выбрать в датафрейме `df` определенный столбец и лишь те строки, что удовлетворяют условию. То есть `i`-я строка будет выбрана, только если `условие[i] = True`.

Команда `(df["married"] == 1) & (df["work"] == 0)` вернет логический вектор `v` (условие), такой, что:


*   `v[i]=True`, если `(df.loc[i, "married"] == 1) & (df.loc[i, "work"] == 0)`, то есть `i`-й индивид состоит в браке и безработен
*   `v[i]=False`, в противном случае

Чтобы выбрать все строки, достаточно использовать команду `df.loc[:, "название столбца"]`.

In [ ]:
# Совместное распределение, условное на дефолт
pd.crosstab([df["default"], df["married"]], df["work"])

In [ ]:
# Оценим вероятности с помощью Байесовского классификатора
  # P(Дефолт = 1 | Брак = 1, Работа = 1)
p11 = np.mean(df.loc[(df["married"] == 1) & (df["work"] == 1), "default"])
  # P(Дефолт = 1 | Брак = 1, Работа = 0)
p10 = np.mean(df.loc[(df["married"] == 1) & (df["work"] == 0), "default"])
  # P(Дефолт = 1 | Брак = 0, Работа = 1)
p01 = np.mean(df.loc[(df["married"] == 0) & (df["work"] == 1), "default"])
  # P(Дефолт = 1 | Брак = 0, Работа = 0)
p00 = np.mean(df.loc[(df["married"] == 0) & (df["work"] == 0), "default"])

Прогнозирование:

$$\widehat{\text{Default}} = \begin{cases}1\text{, если }p_{ab}\geq c\\0\text{, в противном случае}\end{cases}$$

Для простоты рассмотрим порог $c=0.5$:

*   если $p_{ab}\geq 0.5$, то условная вероятность дефолта не меньше, чем его отсутствия, поэтому прогнозируем дефолт $\widehat{\text{Default}} = 1$.
*   если $p_{ab}< 0.5$, то условная вероятность дефолта меньше, чем его отсутствия, поэтому прогнозируем отсутствие дефолта $\widehat{\text{Default}} = 0$.

Например:

*   если $p_{10}\geq 0.5$ то спрогнозируем, что у состоящих в браке $\text{Married}=1$ безработных $\text{Work}=0$ индивидов случится дефолт $\widehat{\text{Default}} = 1$.
*   если $p_{01}< 0.5$ то спрогнозируем, что у холостых $\text{Married}=0$ работающих $\text{Work}=1$ индивидов не случится дефолт $\widehat{\text{Default}} = 0$.


In [ ]:
# Оценки условных вероятностей
print(pd.DataFrame(data    = np.array([np.round([p11, p10, p01, p00], 3),
                                      ['p11', 'p10', 'p01', 'p00']]).transpose(),
                   index   = ['P(Дефолт = 1 | Брак = 1, Работа = 1)',
                              'P(Дефолт = 1 | Брак = 1, Работа = 0)',
                              'P(Дефолт = 1 | Брак = 0, Работа = 1)',
                              'P(Дефолт = 1 | Брак = 0, Работа = 0)'],
                   columns = ['Оценка', 'Переменная']))

Разделим данные на две части:

*   `target` - вектор значений целевой переменной $Y$.
*   `features` - матрица признаков $X$.





In [ ]:
# Разделим целевую переменную и признаки
target   = df.loc[:, ['default']]            # целевая переменная
features = df.loc[:, ["married", "work"]]    # матрица признаков
target   = np.squeeze(target)                # преобразуем из вектора столбца
                                             # в одномерный массив

Чтобы получить прогнозы дефолтов среди индивидов, необходимо для кажого из них знать вероятность дефолта $p_{ab}$, где $a$ и $b$ зависят от фактических значений $\text{Married}_{i}$ и $\text{Work}_{i}$.

Например, если $i$-й индивид женат и безработен, то $\text{Married}_{i}=1$ и $\text{Work}_{i}=0$, а значит его условная вероятность дефолта равняется $p_{10}$. Следовательно,при $p_{10}\geq0.5$ мы спрогнозируем $\widehat{\text{Default}}_{i}=1$.

Для удобства сформируем вектор, состоящий из условных вероятностей `prob_b`, где:

 `prob_b[i]`$=\hat{\text{P}}(\text{Default} = 1| \text{Married} = \text{Married}_{i}, \text{Work} = \text{Work}_{i})$

Техническое примечание ⚡

Команда `x[a:b]` позволяет отобрать элементы массива `x` с `a`-го по `(b-1)`-й. Например, `x[2:5]` отберет элементы с индексами со `2`-го по `4`-й включительно (важно помнить, что индексы начинаются с `0`).

Дополнительная информация может быть найдена в [документации](https://python-reference.readthedocs.io/en/latest/docs/brackets/slicing.html).

In [ ]:
# Назначим каждому наблюдению соответствующую ему
# вероятность P(Дефолт = 1 | Брак, Работа)
prob_b = np.zeros(n)
prob_b[(df["married"] == 1) & (df["work"] == 1)] = p11
prob_b[(df["married"] == 1) & (df["work"] == 0)] = p10
prob_b[(df["married"] == 0) & (df["work"] == 1)] = p01
prob_b[(df["married"] == 0) & (df["work"] == 0)] = p00

# Условные вероятности дефолта для первых нескольких индивидов
print(prob_b[0:10])

Создадим вектор прогнозов дефолтов Байесовского классификатора `prediction_b`, где:

`prediction_b[i]`$=\begin{cases}1\text{, если }\text{prob}\_\text{b[i]}\geq0.5\\0\text{, в противном случае}\end{cases}$



Техническое примечание ⚡

Команда `prob_b >= 0.5` вернет логический вектор `v`, такой, что:


*   `v[i]=True`, если `prob_b[i]>=0.5`
*   `v[i]=False`, если `prob_b[i]<0.5`

Затем, с помощью команды `np.array(v, dtype = int)`, значения `True` в `v` заменяются на `1`, а `False` на `0`.

In [ ]:
# Спрогнозируем Дефолт тем, у кого вероятность не меньше 0.5
prediction_b = (prob_b >= 0.5).astype(int)

# Прогнозы дефолта для первых нескольких индивидов
print(prediction_b[0:10])

Оценим точность Байесовского классификатора как долю верно спрогнозированных дефолтов:

$$\text{ACC} = \frac{1}{n}\sum\limits_{i=1}^{n}I(\text{Default}_{i} = \widehat{\text{Default}_{i}})$$

In [ ]:
# Оценим точность внутривыборочного прогноза
# Байесовского классификатора
ACC_b = np.mean(target == prediction_b)
print(ACC_b)

**Наивный байесовский классификатор: ручная реализация на простом примере** 🐱

**Цели** ⭐

*   С помощью навиного байесовского классификатора оценить одну условную вероятность дефолта (остальные оцениваются по аналогии).

Воспользуемся наивным Байесовским классификатором, чтобы оценить одну из условных вероятностей. Напомним, что в силу допущения об условной независимости случайные величины $\text{Married}$ и $\text{Work}$ предполагаются независимыми при условии $\text{Default}$, то есть:

$$P(\text{Married} = a, \text{Work} = b | \text{Default} = c) = P(\text{Married} = a| \text{Default} = c)P(\text{Work} = b | \text{Default} = c)$$

Применяя этой свойство и формулу условной вероятности получаем:

 $$P(\text{Default} = 1 | \text{Married} = 1, \text{Work} = 0) = \frac{P(\text{Default} = 1)P(\text{Married} = 1, \text{Work} = 0|\text{Default} = 1)}{P(\text{Married} = 1, \text{Work} = 0)} = \frac{p_{1}}{p_{1} + p_{0}}$$

 Где:

$$p_{1} = P(\text{Default} = 1)P(\text{Married} = 1|\text{Default} = 1)P(\text{Work} = 0|\text{Default} = 1)$$

$$p_{0} = P(\text{Default} = 0)P(\text{Married} = 1|\text{Default} = 0)P(\text{Work} = 0|\text{Default} = 0)$$

Сперва оценим априорную вероятность $P(\text{Default} = 1)$.

In [ ]:
# Оценим априорную вероятность P(Дефолт = 1)
p_default = np.mean(df["default"])
print(p_default)

Оценим факторы (маржинальные условные вероятности)


*   $P(\text{Married} = 1|\text{Default} = 1)$
*   $P(\text{Work} = 0|\text{Default} = 1)$
*   $P(\text{Married} = 1|\text{Default} = 0)$
*   $P(\text{Work} = 0|\text{Default} = 0)$





In [ ]:
# P(Брак = 1 | Дефолт = 1)
p_m1d1 = np.mean(df.loc[df["default"] == 1, "married"])

# P(Работа = 0 | Дефолт = 1) = 1 - P(Работа = 1 | Дефолт = 1)
p_w0d1 = 1 - np.mean(df.loc[df["default"] == 1, "work"])

# P(Брак = 1 | Дефолт = 0)
p_m1d0 = np.mean(df.loc[df["default"] == 0, "married"])

# P(Работа = 0 | Дефолт = 0) = 1 - P(Работа = 1 | Дефолт = 0)
p_w0d0 = 1 - np.mean(df.loc[df["default"] == 0, "work"])

Представим искомую вероятность как:
$$P(\text{Default} = 1 | \text{Married} = 1, \text{Work} = 0) = \frac{p_{1}}{p_{1}+p_{0}}$$

где:

*   $p_{1} = P(\text{Default} = 1)P(\text{Married} = 1|\text{Default} = 1)P(\text{Work} = 0|\text{Default} = 1)$
*   $p_{0} = P(\text{Default} = 0)P(\text{Married} = 1|\text{Default} = 0)P(\text{Work} = 0|\text{Default} = 0)$



In [ ]:
# Оценим вспомогательные вероятности
p1 = p_default       * p_m1d1 * p_w0d1
p0 = (1 - p_default) * p_m1d0 * p_w0d0

# Оценим P(Дефолт = 1 | Брак = 1, Работа = 0)
p_d1_m1w0 = p1 / (p1 + p0)

# Если условная вероятность дефолта не меньше 0.5, то прогнозируем дефолт для
# безработных людей в браке, а в противном случае - отсутствие дефолта
print(p_d1_m1w0)

Остальные условные вероятности могут быть оценены по аналогии.

**Наивный байесовский классификатор: автоматическая реализация** 🐱

**Цели** ⭐

*   Оценить условную вероятность дефолта для каждого индивида в выборке c помощью наивного байесовского классификатора.  
*   Используя оцененные вероятности спрогнозировать дефолт.
*   Посчитать точность прогноза $\text{ACC}$.


Техническое примечание ⚡

Функция `CategoricalNB()` возвращает объект, который можно использовать для обучения наивного байесовского классификатора, с целью последующего прогнозирования и оценивания качества модели.

С помощью кода `nb = CategoricalNB(force_alpha = True, alpha = 0)` мы создаем наивный байесовский классификатор `nb`. Далее используя код `nb.fit(features, target)` мы обучаем этот классификатор, то есть оцениваем условные вероятности с помощью данных о признаках `features` и значениях целевой переменной `target`.

Функция `CategoricalNB()` именуется **конструктором**, поскольку создает **экземпляр класса** `CategoricalNB`, который мы назвали `nb`. Функция `.fit()` называется **методом**, поскольку вызывается экземпляром класса `nb`.

Подробней о функции `CategoricalNB()` можно прочитать в [документации](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.CategoricalNB.html)

In [ ]:
# Обучим наивный Байесовский классификатор
# с помощью встроенной функции
nb = CategoricalNB(force_alpha = True,     # подготавливаем модель
                   alpha       = 0)        # alpha - параметр сглаживания Лапласа
nb.fit(features, target)                   # обучаем модель

Оценим условные вероятности:

*   $P(\text{Default} = 0 | \text{Married} = \text{Married}_{i}, \text{Work} = \text{Work}_{i})$
*   $P(\text{Default} = 1 | \text{Married} = \text{Married}_{i}, \text{Work} = \text{Work}_{i})$


In [ ]:
# Оценим условные вероятности
prob_nb = nb.predict_proba(features)
print(prob_nb[0:10:, 0])                   # оценки P(Y = 0 | X = x)
print(prob_nb[0:10:, 1])                   # оценки P(Y = 1 | X = x)

Спрогнозируем дефолты:

$$\widehat{\text{Default}}_{i} = I(\hat{P}(\text{Default} = 1 | \text{Married} = \text{Married}_{i}, \text{Work} = \text{Work}_{i})\geq 0.5)=\begin{cases}1\text{, если }\hat{P}(\text{Default} = 1 | \text{Married} = \text{Married}_{i}, \text{Work} = \text{Work}_{i})\geq 0.5\\0\text{, в противном случае}\end{cases}$$

In [ ]:
# Прогнозы
prediction_nb = nb.predict(features)       # I(P(Y = 1 | X = x) >= 0.5)
print(prediction_nb[0:10])

Оценим точность наивного байесовского классификатора как долю верно спрогнозированных дефолтов:

$$\text{ACC} = \frac{1}{n}\sum\limits_{i=1}^{n}I(\text{Default}_{i} = \widehat{\text{Default}_{i}})$$

Технический комментарий ⚡

Метод `score()` класса `CategoricalNB` позволяет посчитать точность модели.

In [ ]:
# Оценим точность наивного Байесовского
# класификатора внутривыборочно
ACC_nb = np.mean(target == prediction_nb) # вручную как долю случаев, когда
                                          # значение целевой переменной совпало
                                          # с прогнозом
ACC_nb = nb.score(features, target)       # автоматически

# Сравним точность Байесовского классификатора и
# Наивного Байесовского классификатора
print(pd.DataFrame(data    = [ACC_b, ACC_nb],
                   index   = ['Байесовский классификатор',
                              'Наивный Байесовский классификатор'],
                   columns = ['ACC']))

Оценим условную вероятность:
$$P(\text{Default} = 1 | \text{Married} = 1, \text{Work} = 0)$$

In [ ]:
# Оценим вероятность для конкретного наблюдения
observation = pd.DataFrame(data = {'married': [1], # Брак = 1
                                   'work': [0]})   # Работа = 0
p_observation = nb.predict_proba(observation)
print(p_observation[:, 0])                         # P(Дефолт = 0 | Брак = 1, Работа = 0)
print(p_observation[:, 1])                         # P(Дефолт = 1 | Брак = 1, Работа = 0)

In [ ]:
# Сравним результаты ручных расчетов с автоматической функцией
print(pd.DataFrame(data    = [p_d1_m1w0, p_observation[0, 1]],
                   index   = ['Ручной расчет (наш)',
                              'Автоматический расчет (функцией)'],
                   columns = ['Оценка P(Дефолт = 1 | Брак = 1, Работа = 0)']))

**Сопоставление результатов Байесовского классификатора и наивного Байесовского классификатора** 🐱

**Цели** ⭐

*   Сравнить оценки условных вероятностей и прогнозы Байесовского классификатор и наивного Байесовского классификатора.


In [ ]:
# Создадим таблицу с результатами,
# содержащую данные, вероятности
# и прогнозы
table = features.copy()                 # признаки
table["default"]       = target         # целевая переменная
table["prediction_b"]  = prediction_b   # прогнозы Байесовского классификатора
table["prob_b"]        = prob_b         # вероятности Байесовского
                                        # классификатора
table["prediction_nb"] = prediction_nb  # прогнозы наивного Байесовского
                                        # классификатора
table["prob_nb"]       = prob_nb[:, 1]  # вероятности наивного Байесовского
                                        # классификатора
table.head(10)

**Обучение наивного Байесовского классификатора с дополнительным признаком на уровень образования** 🐱

**Цели** ⭐

*   Измерить изменение точности прогнозов наивного байесовского классификатора после добавления дополнительной переменной.


In [ ]:
# Изучим возможность добавления признака образования
pd.crosstab([df["married"], df["work"]], df["educ"])

**Проблема** - некоторые комбинации признаков наблюдаются крайне редко, что осложняет применение Байесовский классификатора, по крайней мере без сглаживания

**Решение** - воспользоваться наивным Байесовским классификатором

In [ ]:
# Повторим оценивание наивного Байесовского
# классификатора добавив признак на
# уровень образования

# Копируем признаки исходной модели
features2 = features.copy()

# Добавляем признак образования
features2["educ"] = df["educ"]

# Подготавливаем новую модель
nb2 = CategoricalNB(alpha = 0, force_alpha = True)

# Обучаем модель
nb2.fit(features2, target)

# Достаем прогнозы новой модели
prediction2 = nb2.predict(features2)

# Считаем точность новой модели
ACC_nb2 = nb2.score(features2, target)

# Сравниваем точность прогнозов до и после
# добавления переменной на образование
print(pd.DataFrame(data    = [ACC_nb, ACC_nb2],
                   index   = ['Модель, не учитывающая образование',
                              'Модель, учитывающая образование'],
                   columns = ['ACC']))

**Анализ точности наивного Байесовского классификатора с применением тестовой выборки** 🐱

**Цели** ⭐

*   Разбить выборку на обучющую и тестовую.
*   Сравнить точность прогнозов навиного байесовского классификатора на обучающей и тестовой выборках.


Технический комментарий ⚡

Функция `train_test_split()` позволяет разбить выборку на обучающую и тестовую.

Основные аргументы:

*   `*arrays` - данные (записываются через запятую), которые необходимо разбить на обучающую и тестовую части.
*   `test_size` - доля тестовой выборки, например, если `test_size = 0.2` и у вас 100 наблюдений, то в тестовую выборку войдет 20 наблюдений, а в обучающую попадет 80 наблюдений.
*   `random_state` - параметр, позволяющий изменить наблюдения, случайным образом попадающие в тестовую и обучающую выборки.

Дополнительную информацию можно найти в [документации](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

Если бы нам было необходимо разбить на обучающую и тестовую части лишь данные, касающие признаков `features`, то мы бы вызывали код как:

`features_train, features_test = train_test_split(features, test_size = 0.2, random_state = 777)`

Обратите внимание, что функция возвращает сразу два набора данных:

*   `features_train` - обучающая выборка признаков.
*   `features_test` - тестовая выборка признаков.

Однако, мы используем код:

`features_train, features_test, target_train, target_test = train_test_split(
features, target, test_size = 0.2, random_state = 777)`

В качестве аргумента `*arrays` мы указываем сразу два набора данных: `features` и `target`. Поэтому и возвращает функция уже не два, а четыре набора данных:

*   `features_train` - обучающая выборка признаков.
*   `features_test` - тестовая выборка признаков.
*   `target_train` - обучающая выборка значений целевой переменной.
*   `target_test` - тестовая выборка значений целевой переменной.



In [ ]:
# Разделим выборку на обучающую и тестовую
# с помощью автоматической функции
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size = 0.2, random_state = 777)

# Убедимся, что обучающая и тестовая выборки имеют верные пропорции
print(features_train.index.size, features_test.index.size) # признаки
print(target_train.index.size, target_test.index.size)     # целевая переменная

Технический комментарий ⚡

Объекты `nb` и `nb_train` являются экземплярами одного и того же класса `CategoricalNB`. Однако эти экземпляры различаются между собой тем, что первый представляет собой наивный байесовский классификатор, обученный на всей выборке `nb.fit(features, target)`, а второй - только на обучающей `nb_train.fit(features_train, target_train)`.

In [ ]:
# Оценим первую модель (без образования) на обучающей выборке
nb_train = CategoricalNB(force_alpha = True, alpha = 0)
nb_train.fit(features_train, target_train)

Технический комментарий ⚡

Обратите внимание, что для оценивания точность модели `nb_train` на тестовой выборке, метод `score()` используется с тестовой выборкой, то есть аргументами `features_test` и `target_test`.

In [ ]:
# Оценим точность прогноза на тестовой выбокре
ACC_nb_test = nb_train.score(features_test, target_test)

In [ ]:
# Сравним точность прогноза на обучающей и тестовой выборках
ACC_nb_train = nb_train.score(features_train, target_train)
print(pd.DataFrame(data    = [ACC_nb_train, ACC_nb_test],
                   index   = ['Обучающая выборка',
                              'Тестовая выборка'],
                   columns = ['ACC']))

**Кросс-валидация наивного Байесовского классификатора** 🐱

**Цели** ⭐

*   Посчитать точность прогнозов наивного байесовского классификатора с помощью кросс-валидации.
*   Сравнить результаты моделей с образованием и без образования.


Технический комментарий ⚡

Функция `cross_val_score()` позволяет посчитать точность с помощью кросс-валидации.

Основные аргументы

*   `estimator` - модель, для которой осуществляется кросс-валидация, например, экземпляр класса `CategoricalNB`.
*   `X` - двумерный массив признаков.
*   `y` - одномерный массив со значениями целевой переменной.
*   `cv` - количество частей (folds), используемых при кросс-валидации.

Функция возвращает одномерный массив, `i`-й элемент которого отражает точность, посчитанную по `i`-й части (фолду) с помощью модели, обученной на всех частях, кроме `i`-й.

Дополнительную информацию можно найти в [документации](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html).



In [ ]:
# Первая модель (без образования)
ACC_CV_nb = cross_val_score(estimator = nb,     # модель
                            X = features,       # признаки
                            y = target,         # целевая переменная
                            cv = 5)             # количество частей (folds)
ACC_CV_total_nb = np.mean(ACC_CV_nb)            # средняя точность по фолдам

# Вектор точностей посчитанны для
# каждой части выборки
print(ACC_CV_nb)

In [ ]:
# Вторая модель (с образованием)
ACC_CV_nb2 = cross_val_score(nb2,               # модель
                             features2, target, # признаки и целевая переменная
                             cv = 5)            # количество фолдов (folds)
ACC_CV_total_nb2 = np.mean(ACC_CV_nb2)          # средняя точность по фолдам

# Вектор точностей посчитанны для
# каждой части выборки
print(ACC_CV_nb2)

In [ ]:
# Сопоставим результаты
print(pd.DataFrame(data    = [ACC_CV_total_nb, ACC_CV_total_nb2],
                   index   = ['Модель, не учитывающая образование',
                              'Модель, учитывающая образование'],
                   columns = ['ACC-CV']))

**Использование биннинга в отношении непрерывной переменной** 🐱

**Цели** ⭐

*   Обучить наивный байесовский классификатор с использованием дополнительной переменной, измеренной в непрерывной шкале, предварительно приведя ее к порядковой шкале.

Напомним, что **биннинг** предполагает превращение непрерывной переменной в дискретную. Без этого применение Байесовского или наивного Байесовского классификатора может оказаться затруднительным, поскольку непрерывные переменные принимают слишком много различных значений, что приводит к существенной фрагментации данных и, как следствие, к проклятью размерности.

Превратим непрерывную переменную доход `income`, в порядковую переменную `incom_bin` разбив доходы на квантилями уровня $1/3$ и $2/3$.

In [ ]:
# Для добавления переменной на доход разобьем ее по квантилям
df['income_bin'] = pd.qcut(df['income'],   # разбиваемая переменная
                           q = 3)          # число квантилей (равных разбиений)
df['income_bin'].value_counts()            # распределение значений


In [ ]:
# Преобразуем переменную в числовую за счет
# добавления аргумента labels
df['income_bin'] = pd.qcut(df['income'], q = 3, labels = range(0, 3))
print(df.loc[0:10, 'income_bin'])


In [ ]:
# Копируем признаки исходной модели,
# учитывавшей уровень образования
features3 = features2.copy()

# Добавляем признак дохода
features3["income_bin"] = df["income_bin"]

# Подготавливаем новую модель
nb3 = CategoricalNB(force_alpha = True, alpha = 0)

# Обучаем модель
nb3.fit(features3, target)

# Достаем прогнозы новой модели
prediction3 = nb3.predict(features3)

# Считаем точность новой модели
ACC_nb3          = nb3.score(features3, target)     # внутривыборочная
ACC_CV_nb3       = cross_val_score(nb3, features3,  # кросс-валидация
                                   target, cv = 5)
ACC_CV_total_nb3 = np.mean(ACC_CV_nb3)


# Сопоставим результаты
print(pd.DataFrame(data    = [ACC_CV_total_nb,
                              ACC_CV_total_nb2,
                              ACC_CV_total_nb3],
                   index   = ['Модель, не учитывающая образование и доход',
                              'Модель, учитывающая образование',
                              'Модель, учитывающая образование и доход'],
                   columns = ['ACC-CV']))

**Байесовская сеть** 🐱

**Цели** ⭐

*   Сформировать DAG, удовлетворяющий содержательным предпосылкам о связи между переменными.
*   С помощью созданного DAG обучить Байесовскую сеть.
*   Оценить условные вероятности дефолтов с помощью Байесовской сети.
*   Используя оцененные вероятности спрогнозировать вероятности дефолтов.
*   Оценить точность прогнозов $\text{ACC}$ Байесовской сети.

Структура байесовской сети определяется направленным ацикличным графом (DAG - directed acyclic graph). Поэтому, сперва необходимо сформировать DAG, указав предполагаемые направления причинно-следственных связей между переменными.

Технический комментарий ⚡

Прежде, чем сформировать DAG, необходимо создать матрицу `edges`, отражающую связи между переменными. Например, ее элемент `("educ", "income_bin")` указывает, что образование `educ` является причиной `income_bin` для дохода. Это связь будет отражена в DAG в форме стрелочки от `educ` к `income_bin`. Затем матрица `edges` прерващается в `DAG` с помощью функции `bnlearn.make_DAG()`.

Подробная информация о различных способах задать DAG может быть найдена в [документации](https://erdogant.github.io/bnlearn/pages/html/Create%20DAG.html#building-a-causal-dag).

In [ ]:
# Создадим DAG
edges = [("work", "income_bin"),     # (откуда стрелочка, куда стрелочка)
         ("educ", "income_bin"),
         ("income_bin", "default"),
         ("educ", "default"),
         ("work", "default"),
         ("married", "default")]
DAG   = bnlearn.make_DAG(edges, methodtype = 'bayes')
if 'config' not in DAG:
    DAG['config'] = {'method': 'bayes'}

[bnlearn] >bayes DAG created.


Технический комментарий ⚡

Вместо того, чтобы создавать экземпляр класса, сразу создадим переменную, содержащую обученную модель.

Подробней о пакете `bnlearn` можно узнать в [документации](https://erdogant.github.io/bnlearn/pages/html/index.html).

In [ ]:
# Оценим факторы
bn = bnlearn.parameter_learning.fit(DAG, df, methodtype = 'ml')

In [ ]:
# Визуализируем DAG
bnlearn.plot(bn, interactive = False)

Технический комментарий ⚡

Функция `bnlearn.predict()` прогнозирует значения и оценивает условные вероятности переменной (или совместную условную вероятность для нескольких переменных) `variables` с использованием данных (условий) `df` и факторов, оцененных моделью (первый аргумент функции).

Функция возвращает датафрейм с двумя столбцами. Первый столбец называется так же, как и `variables`. Например, если целевая переменная называлась `default`, то и столбец будет называться `default`. Этот столбец содержит прогнозы $\hat{Y}$ соответствующей переменной (наиболее вероятные категории). Второй столбец `p` содержит условные вероятности $\hat{\text{P}}(Y|X)$.

**Важно** - функция возвращает вектор условных вероятностей `p` наиболее вероятных категорий, то есть:

`p[i]` = $\begin{cases}\hat{\text{P}}(Y_{i}=1|X_{i})\text{, если }\hat{\text{P}}(Y_{i}=1|X_{i})\geq \hat{\text{P}}(Y_{i}=0|X_{i})\\ \hat{\text{P}}(Y_{i}=0|X_{i})\text{, в противном случае}\end{cases}$

Подробней об этой функции можно узнать в [документации](https://erdogant.github.io/bnlearn/pages/html/Predict.html).

In [ ]:
# Оценим условные вероятности дефолта
predict_bn = bnlearn.predict(bn, df = features3, variables = ["default"])

# Посмотрим на матрицу.
predict_bn.head(10)

In [ ]:
# Достанем прогнозы Дефолтов
prediction_bn = predict_bn.loc[:, "default"]

# Достанем оценки условных вероятностей дефолта
# с учетом того, что изначально мы имеем лишь вероятности
# наиболее вероятных категорий
prob_nb = np.zeros(len(predict_bn.loc[:, "p"]))
prob_nb[prediction_bn == 0] = 1 - predict_bn.loc[prediction_bn == 0, "p"]
prob_nb[prediction_bn == 1] = predict_bn.loc[prediction_bn == 1, "p"]

In [ ]:
# Оценим точность на обучающей выборке
ACC_bn = np.mean(target == prediction_bn)

# Сравнение точностей на обучающей выборке
print(pd.DataFrame(data    = [ACC_nb3, ACC_bn],
                   index   = ['Наивный Байесовский классификатор',
                              'Байесовская сеть'],
                   columns = ['ACC']))

**Проверка точности Байесовской сети на тестовой выборке и сравнение с навиным Байесовским классификатором** 🐱

**Цели** ⭐

*   Разбить выборку на обучающую и тестовую.
*   Обучить байесовскую сеть на обучающей выборке.
*   Спрогнозировать дефолты на тестовой выборке, используя байесовскую сеть, обученную на обучающей выборке.
*   Оценить точность прогнозов $\text{ACC}$ байесовской сети на тестовой выборке.

In [ ]:
# Разобьем выборку на обучающую и тестовую
features3_train, features3_test, target_train, target_test = train_test_split(
    features3, target, test_size = 0.2, random_state = 777)

In [ ]:
# Поскольку для обучения Байесовской сети нужно подавать сразу все
# данные, без разбиения на признаки и целевую переменную, агрегируем
# их в единный датафрейм
df_train            = features3_train.copy()
df_train["default"] = target_train

In [ ]:
# Оценим параметры модели на обучающей выборке
bn_train = bnlearn.parameter_learning.fit(DAG, df_train, methodtype = 'ml')

In [ ]:
# Получим прогнозы по тестовой выборке
predict_bn_test = bnlearn.predict(bn, df = features3_test,
                                  variables = ["default"])
prediction_bn_test = np.array(predict_bn_test.loc[:, "default"])

In [ ]:
# Оценим точность прогноза по тестовой выборке и сравним
# с внутривыборочным прогнозом
ACC_bn_test = np.mean(target_test == prediction_bn_test)

# Сравнение точностей на обучающей и тестовой выборках
print(pd.DataFrame(data    = [ACC_bn, ACC_bn_test],
                   index   = ['Обучающая выборка',
                              'Тестовая выборка'],
                   columns = ['ACC']))

In [ ]:
# Оценим точность на тестовой выборке
# наивного Байесовского классификатора
nb3.fit(features3_train, target_train)                       # обучение на тесте
prediction3_test = nb3.predict(features3_test)               # прогнозы
ACC_nb3_test     = nb3.score(features3_test, target_test)    # точность на тесте

# Сравним точность на тестовой выборке наивного Байесовского
# классификатор и Байесовской сети
print(pd.DataFrame(data    = [ACC_nb3_test, ACC_bn_test],
                   index   = ['Наивный Байесовский классификатор',
                              'Байесовская сеть'],
                   columns = ['ACC-test']))

**Кросс-валидация Байесовской сети** 🐱

**Цели** ⭐

*   Реализовать кросс-валидацию Байесовской сети.

Часто в различных пакетах не реализована автоматическая процедура кросс-валидации, поэтому ее необходимо запрограммировать самостоятельно.

Технический комментарий ⚡

На самом деле компьютер генерирует случайные числа не случайным образом, а отталкиваясь от некоторого первоначального числа, случайность которого часто гарантируется внешним образом, например, временем или температурой процессора.

Если это исходное число задать вручную, то все случайные числа при повторении эксперимента будут совпадать. С этой целью применяется функция `set.seed()`, в качестве аргумента которой указывается соответствующее начальное число **seed**.

In [ ]:
# Для воспроизводимости
random.seed(123)

In [ ]:
# Реализуем 5-частную кросс-валидацию
n_folds = 5

Технический комментарий ⚡

Функция `shuffle()` переставляет наблюдения в векторе в случайном порядке. Благодаря этой функции мы можем отсортировать наблюдения в случайном порядке, что гарантирует случайное распределение наблюдений между частями (folds).

In [ ]:
# Рандомизируем порядок наблюдений
ind_random = np.array(shuffle(range(0, n)))

In [ ]:
# Вектор, в котором будут храниться результаты кросс-валидации
ACC_CV_bn = np.zeros(n_folds)

In [ ]:
# Создадим массив, i-й элемент которого содержит
# индексы наблюдений, попавших в i-ю часть выборки (fold)
ind_fold = np.split(ind_random, n_folds)

In [ ]:
# Пройдемся циклом отдельно по каждому из фолдов
for i in range(0, n_folds):
  # Тестовая выборка
  features_test_cv = features3.iloc[ind_fold[i]]
  target_test_cv = target[ind_fold[i]]
  # Обучающая выборка
  features_train_cv = features3.iloc[~df.index.isin(ind_fold[i])]
  target_train_cv = target[~df.index.isin(ind_fold[i])]
  # Объединение признаков и целевой переменной обучающей выборки
  df_train_cv = features_train_cv
  df_train_cv["default"] = target_train
  # Оцениваем параметры на обучающей выборке
  bn_train_cv = bnlearn.parameter_learning.fit(DAG, df_train_cv,
                                               methodtype = 'ml')
  # Получим прогнозы по тестовой выборке
  predict_bn_test_cv = bnlearn.predict(bn_train_cv, df = features_test_cv,
                                       variables = ["default"])
  prediction_bn_test_cv = np.array(predict_bn_test_cv.loc[:, "default"])
  # Оценим точность прогноза по тестовой выборке и сравним
  # с внутривыборочным прогнозом
  ACC_CV_bn[i] = np.mean(target_test_cv == prediction_bn_test_cv)

In [ ]:
# Посмотрим на результаты по фолдам
print(ACC_CV_bn)

In [ ]:
# Усредним результаты
ACC_CV_total_bn = np.mean(ACC_CV_bn)
print(ACC_CV_total_bn)

**Обучение структуры Байесовской сети** 🐱

**Цели** ⭐

*   Подобрать оптимальный DAG.
*   Оценить байесовскую сеть с подобранным оптимальным DAG.
*   Сравнить точность прогнозов $\text{ACC}$ исходной байесовской сети (с DAG, сформированных исходя из теоретических соображений) и новой (с оптимальным DAG).

Технический комментарий ⚡

Для того, чтобы подобрать оптимальную структуру (DAG) Байесовской сети, воспользуемся функцией `bnlearn.structure_learning.fit()` со следующими основными аргументами:

*   `methodtype` - метод поиска (обучения) оптимальной структуры Байесовской сети. Например, значение `'hc'` соответствует hill climb search.
*   `scoretype`  - критерий качества структуры сети, используемый при поиске ее оптимальной формы. Например, `'bic'` соответствует информационному критерию BIC, а `'aic'` - информационному критерию AIC.

К сожалению, в данной функции нельзя самостоятельно задать начальный DAG и в качестве него по умолчанию используется граф без стрелочек.

Подробная информация об особенностях обучения структуры Байесовский сети может быть найдена в [документации](https://erdogant.github.io/bnlearn/pages/html/Structure%20learning.html).

In [ ]:
# Подберем оптимальную структуру DAG на обучающей выборе
bn2_train_structure = bnlearn.structure_learning.fit(df_train,
                                                     methodtype = 'hc',
                                                     scoretype  = 'bic')

In [ ]:
# Посмотрим на результат
bnlearn.plot(bn2_train_structure, interactive = False)

In [ ]:
# Сохраним найденный DAG
DAG2 = bnlearn.make_DAG(bn2_train_structure)
if 'config' not in DAG2:
    DAG2['config'] = {'method': 'bayes'}

In [ ]:
# Оценим модель с подобранным DAG
bn2_train = bnlearn.parameter_learning.fit(DAG2, df_train,
                                           methodtype = 'ml')

In [ ]:
# Получим прогнозы по тестовой выборке
predict_bn2_test = bnlearn.predict(bn2_train, df = features3_test,
                                   variables = ["default"])
prediction_bn2_test = np.array(predict_bn2_test.loc[:, "default"])

In [ ]:
# Оценим точность прогноза по тестовой выборке
ACC_bn2_test = np.mean(target_test == prediction_bn2_test)

# Сравним точность Байесовской сети с
# исходным и обученным DAG
print(pd.DataFrame(data    = [ACC_bn_test, ACC_bn2_test],
                   index   = ['Байесовская сеть с исходным DAG',
                              'Байесовская сеть с обученным DAG'],
                   columns = ['ACC-test']))

**Ручной расчет вероятностей в Байесовской сети** 🐱

**Цели** ⭐

*   Вручную оценить условную вероятность в Байесовской сети.
*   Сопоставить результаты ручных расчетов с `bnlearn`.

Рассмотрим новый пример со следующими переменными:

*   `buy` - покупка.
*   `female` - женщина.
*   `work` - работа.
*   `married` - брак.
*   `health` - хорошее здоровье.



Сформируем данные

In [ ]:
buy       = np.array([1, 0, 1, 0, 1, 1, 1, 0, 0])
female    = np.array([1, 1, 1, 1, 0, 1, 0, 1, 0])
work      = np.array([1, 0, 1, 1, 0, 1, 1, 0, 0])
married   = np.array([1, 0, 1, 1, 1, 1, 0, 1, 0])
health    = np.array([1, 0, 1, 1, 0, 1, 1, 0, 0])
df2 = pd.DataFrame({'buy' : buy, 'female' : female, 'work' : work,
                    'married' : married, 'health' : health})
print(df2)

Составим графическую модель (DAG)

In [ ]:
# Предположим следующие связи
edges2 = [("female", "work"),
          ("female", "health"),
          ("work", "buy"),
          ("work", "married"),
          ("married", "buy"),
          ("buy", "health")]
DAG3 = bnlearn.make_DAG(edges2)
if 'config' not in DAG3:
    DAG3['config'] = {'method': 'bayes'}
bnlearn.plot(DAG3, interactive = False)

Оценим совместные вероятности с учетом предполагаемых связей (DAG):

$$p_{1} = P(\text{Female} = 1, \text{Work} = 1, \text{Married} = 1, \text{Health} = 1, \text{Buy} = 1)$$

$$p_{0} = P(\text{Female} = 1, \text{Work} = 1, \text{Married} = 1, \text{Health} = 1, \text{Buy} = 0)$$

Эти совместные вероятности оцениваются с помощью факторов:

$$P(\text{Female} = 1)\qquad P(\text{Work} = 1 | \text{Female} = 1)$$

$$P(\text{Married} = 1 | \text{Work} = 1)\qquad P(\text{Health} = 1 | \text{Female} = 1, \text{Buy} = 1)$$

$$P(\text{Health} = 1 | \text{Female} = 1, \text{Buy} = 0)\qquad P(\text{Buy} = 1 | \text{Work} = 1, \text{Married} = 1)$$

In [ ]:
p_f  = np.mean(female)
p_w  = np.mean(work[female == 1])
p_m  = np.mean(married[work == 1])
p_h1 = np.mean(health[(female == 1) & (buy == 1)])
p_h0 = np.mean(health[(female == 1) & (buy == 0)])
p_b  = np.mean(buy[(work == 1) & (married == 1)])
print(pd.DataFrame(data    = [p_f, p_w, p_m, p_h1, p_h0, p_b],
                   index   = ['P(Female = 1)',
                              'P(Work = 1 | Female = 1)',
                              'P(Married = 1 | Work = 1)',
                              'P(Health = 1 | Female = 1, Buy = 1)',
                              'P(Health = 1 | Female = 1, Buy = 0',
                              'P(Buy = 1 | Work = 1, Married = 1)'],
                   columns = ['Оценка фактора']))

Оценим условную вероятность:

$$p = P(\text{Buy} = 1|\text{Female} = 1, \text{Work} = 1, \text{Married} = 1, \text{Health} = 1) = \frac{p_{1}}{p_{0} + p_{1}}$$

In [ ]:
p1 = p_f * p_w * p_m * p_h1 * p_b
p0 = p_f * p_w * p_m * p_h0 * (1 - p_b)
p = p1 / (p0 + p1)
print(pd.DataFrame(data    = [p1, p0, p],
                   index   = ['P(Female = 1, Work = 1, Married = 1, Health = 1, Buy = 1)',
                              'P(Female = 1, Work = 1, Married = 1, Health = 1, Buy = 0)',
                              'P(Buy = 1 | Female = 1, Work = 1, Married = 1, Health = 1)'],
                   columns = ['Оценка вероятности']))

Сравним наш результат с `bnlearn`

In [ ]:
# Обучаем модель
bn3 = bnlearn.parameter_learning.fit(DAG3, df2, methodtype = 'ml', smooth = 0.2)

# Прогнозируем вероятность
evidence = pd.DataFrame({'female' : [1], 'work' : [1],
                         'married' : [1], 'health' : [1]})
p_bnlearn = bnlearn.predict(bn3, df = evidence, method = None, variables = ["buy"])

In [ ]:
print(pd.DataFrame(data    = [p, p_bnlearn["p"]],
                   index   = ['Наша', 'bnlearn'],
                   columns = ['Оценка условной вероятности']))